#Business Insights Summary

This notebook summarizes the key findings from the Target e-commerce analysis and converts them into business insights and recommendations.

The goal is to connect the SQL analysis results with business meaning across:
- order trends
- customer geography
- payment behavior
- freight cost
- delivery performance

In [0]:
%sql
/*
Overall Business Summary

Purpose:
    Summarize the main business metrics from the e-commerce dataset,
    including total orders, customers, payment value, and average order value.
*/

SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT o.customer_id) AS total_customers,
    ROUND(SUM(p.payment_value), 2) AS total_payment_value,
    ROUND(AVG(p.payment_value), 2) AS avg_payment_value
FROM ecommerce_analysis.orders o
LEFT JOIN ecommerce_analysis.payments p
    ON o.order_id = p.order_id;

In [0]:
%sql
/*
Top Revenue States

Purpose:
    Identify the states generating the highest total payment value.
*/

SELECT
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.payment_value), 2) AS total_payment_value
FROM ecommerce_analysis.orders o
LEFT JOIN ecommerce_analysis.customer c
    ON o.customer_id = c.customer_id
LEFT JOIN ecommerce_analysis.payments p
    ON o.order_id = p.order_id
GROUP BY c.customer_state
ORDER BY total_payment_value DESC
LIMIT 10;

In [0]:
%sql
/*
Delivery Performance Summary

Purpose:
    Summarize delivery speed and estimated delivery accuracy.
    Negative values for avg_days_vs_estimate mean orders were delivered earlier than estimated.
*/

SELECT
    ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)), 2) AS avg_delivery_days,
    ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)), 2) AS avg_days_vs_estimate,
    COUNT(DISTINCT order_id) AS delivered_orders
FROM ecommerce_analysis.orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_purchase_timestamp IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;

In [0]:
%sql
/*
Payment Method Summary

Purpose:
    Compare payment methods by order volume and total payment value.
*/

SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM ecommerce_analysis.payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;

## Key Business Insights

1. **E-commerce activity is strongly concentrated in a few regions**
   - A small number of states drive most of the order volume and payment value.
   - This suggests that demand is not evenly distributed across Brazil, and the business depends heavily on major commercial regions.

2. **São Paulo is the strongest market**
   - SP clearly dominates both orders and revenue.
   - This makes it the most important state for marketing, seller operations, delivery planning, and customer retention.

3. **Delivery performance appears better than estimated**
   - On average, orders are delivered earlier than the estimated delivery date.
   - This suggests that delivery estimates may be conservative, which can protect customer expectations but may also make delivery promises look slower than they really are.

4. **Credit card is the main payment method**
   - Credit card payments dominate the platform’s transaction value and order volume.
   - This shows that payment convenience, card reliability, and installment options are important for customer conversion.

5. **Alternative payment methods are used less frequently**
   - UPI, voucher, and debit card payments contribute less compared to credit card.
   - These methods may still be important for specific customer groups, but they are not the main revenue driver.

6. **Freight and delivery should be analyzed regionally**
   - Since customer demand is regionally concentrated, delivery time and freight cost should not be treated as one national average.
   - State-level logistics analysis is more useful for improving customer experience.

## Recommendations

1. **Prioritize high-demand states**
   - Focus marketing, seller support, and logistics resources on the states with the strongest order and revenue contribution.

2. **Protect the São Paulo market**
   - Since SP is the largest business contributor, maintaining fast delivery, strong seller availability, and smooth checkout there should be a priority.

3. **Improve delivery estimate accuracy**
   - If deliveries are consistently earlier than estimated, estimated delivery dates can be adjusted to be more realistic and competitive.

4. **Optimize credit card checkout**
   - Since credit card is the dominant payment method, the checkout flow should be reliable, fast, and optimized for card users.

5. **Use regional logistics strategy**
   - States with higher freight cost or slower delivery should be investigated separately to improve delivery efficiency and customer satisfaction.

6. **Use seasonality for planning**
   - Monthly order patterns can help plan inventory, seller capacity, and promotional campaigns more effectively.